# Homework 2

## Question 1: [IPO] Withdrawn IPOs by Company Type

**What is the total withdrawn IPO value (in $ millions) for the company class with the highest total withdrawal value?**

Source: recently filed IPO list from [iposcoop.com/ipos-recently-filed](https://www.iposcoop.com/ipos-recently-filed/).

### Steps
1. Load the IPO recently filed table with `pandas.read_html()`.
2. Filter to rows where `Expected To Trade` is `Withdrawn`.
3. Classify companies into a `Company Type` based on name patterns.
4. Parse `Price Low` / `Price High` into an `Avg_price`.
5. Clean `Shares (millions)` and `Est $ Vol (millions)` into numeric columns.
6. Compute `Shares_offered_value` (Shares * Avg_price, falling back to Est $ Vol).
7. Group by `Company Type` and sum to find the largest withdrawn value.

### Data Loading

In [1]:
import re

import numpy as np
import pandas as pd
import requests
from io import StringIO

URL = "https://www.iposcoop.com/ipos-recently-filed/"
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    )
}

response = requests.get(URL, headers=HEADERS, timeout=15)
response.raise_for_status()

tables = pd.read_html(StringIO(response.text))
ipo_df = tables[0].copy()

print(ipo_df.shape)
ipo_df.head()

(500, 10)


,File Date,Company,Symbol,Managers,Shares (millions),Price Low,Price High,Est $ Vol (millions),Expected To Trade,SCOOP Rating
0,2026-09-14,ADARx Pharmaceuticals,ADRX,J.P.Morgan/Morgan Stanley/TD Cowen/UBS Investm...,0.00,NaN,NaN,$100.00,TBA,S/O
1,2026-09-14,Bamboo Insurance Services,BMB,J.P.Morgan/Morgan Stanley/Deutsche Bank Securi...,35.00,$18.00,$20.00,$665.00,Wednesday,S/O
2,2026-09-14,Electra Therapeutics,ETRA,Jefferies/TD Cowen/Evercore ISI/Cantor,21.67,$14.00,$16.00,$325.05,Friday,S/O
3,2026-09-11,Accelevation Holdings Corp.,ACCV,Morgan Stanley/J.P. Morgan/Goldman Sachs/Barcl...,0.00,NaN,NaN,$100.00,TBA,S/O
4,2026-09-10,Allarity Acquisition Corp.,ALLNU,Maxim Group,10.00,$10.00,$10.00,$100.00,TBA,S/O


In [2]:
withdrawn_df = ipo_df[ipo_df["Expected To Trade"] == "Withdrawn"].copy()
print("Withdrawn entries:", len(withdrawn_df))
withdrawn_df.head()

Withdrawn entries: 32


,File Date,Company,Symbol,Managers,Shares (millions),Price Low,Price High,Est $ Vol (millions),Expected To Trade,SCOOP Rating
5,2026-09-10,Motive Technologies (Withdrawn),MTVE,J.P.Morgan/Citigroup/Barclays/Jefferies/RBC Ca...,0.0,NaN,NaN,$100.00,Withdrawn,S/O
12,2026-09-04,Idea Tech Holding (Withdrawn),IDTL,R.F. Lafferty & Co.,2.0,$4.00,$5.00,$9.00,Withdrawn,S/O
21,2026-09-01,Hornbeck Offshore Services (Withdrawn),HOS,J.P. Morgan/ Barclays/DNB Markets/Piper Sandle...,0.0,NaN,NaN,$100.00,Withdrawn,S/O
23,2026-08-31,Coolbit Technologies Ltd. (Withdrawn),CBAI,Eddid Securities USA,5.0,$4.00,$5.00,$22.50,Withdrawn,S/O
32,2026-08-27,Timber Road Acquisition Corp. (Withdrawn),TMRDU,Roth Capital Partners/Stone X Financial Inc.,20.0,$10.00,$10.00,$200.00,Withdrawn,S/O


### Company Classification

In [3]:
def classify_company(name: str) -> str:
    """Assign a Company Type based on the first matching pattern (order matters)."""
    n = str(name).lower()

    if "technologies" in n:
        return "Technologies"
    if "acquisition corp" in n or "acquisition corporation" in n or "corp" in n:
        return "Acquisition Corp"
    if "inc" in n or "incorporated" in n:
        return "Inc."
    if "group" in n:
        return "Group"
    if "ltd" in n or "limited" in n:
        return "Limited"
    if "holdings" in n or "holding" in n:
        return "Holdings"
    return "Other"


withdrawn_df["Company Type"] = withdrawn_df["Company"].apply(classify_company)
withdrawn_df["Company Type"].value_counts()

Company Type
Other               8
Limited             8
Acquisition Corp    5
Holdings            4
Technologies        3
Group               3
Inc.                1
Name: count, dtype: int64

### Price Parsing

In [4]:
def extract_number(value) -> float:
    """Extract a numeric value from a price string, e.g. '$8.00' -> 8.0."""
    if pd.isna(value) or str(value).strip() == "-":
        return np.nan
    match = re.findall(r"[\d.]+", str(value).replace(",", ""))
    return float(match[0]) if match else np.nan


withdrawn_df["Price Low Num"] = withdrawn_df["Price Low"].apply(extract_number)
withdrawn_df["Price High Num"] = withdrawn_df["Price High"].apply(extract_number)
withdrawn_df["Avg_price"] = withdrawn_df[["Price Low Num", "Price High Num"]].mean(axis=1)

withdrawn_df[["Price Low", "Price High", "Avg_price"]].head(10)

,Price Low,Price High,Avg_price
5,NaN,NaN,NaN
12,$4.00,$5.00,4.50
21,NaN,NaN,NaN
23,$4.00,$5.00,4.50
32,$10.00,$10.00,10.00
45,$10.00,$10.00,10.00
61,$4.00,$6.00,5.00
105,$8.00,$10.00,9.00
136,$4.00,$4.00,4.00
141,$4.00,$4.50,4.25


### Numeric Conversion

In [5]:
def to_numeric_clean(series: pd.Series) -> pd.Series:
    cleaned = series.astype(str).str.replace(r"[$,]", "", regex=True).replace("-", np.nan)
    return pd.to_numeric(cleaned, errors="coerce")


withdrawn_df["Shares (millions)"] = to_numeric_clean(withdrawn_df["Shares (millions)"])
withdrawn_df["Est $ Vol (millions)"] = to_numeric_clean(withdrawn_df["Est $ Vol (millions)"])

withdrawn_df[["Shares (millions)", "Est $ Vol (millions)"]].head(10)

,Shares (millions),Est $ Vol (millions)
5,0.00,100.00
12,2.00,9.00
21,0.00,100.00
23,5.00,22.50
32,20.00,200.00
45,12.50,125.00
61,3.75,19.00
105,5.60,50.40
136,3.75,15.00
141,3.75,15.94


### Value Calculation

In [6]:
computed_value = withdrawn_df["Shares (millions)"] * withdrawn_df["Avg_price"]

withdrawn_df["Shares_offered_value"] = computed_value.where(
    computed_value.notna(), withdrawn_df["Est $ Vol (millions)"]
)

print("Non-null Shares_offered_value:", withdrawn_df["Shares_offered_value"].notna().sum())
withdrawn_df[["Company", "Company Type", "Shares_offered_value"]].head(10)

Non-null Shares_offered_value: 32


,Company,Company Type,Shares_offered_value
5,Motive Technologies (Withdrawn),Technologies,100.0000
12,Idea Tech Holding (Withdrawn),Holdings,9.0000
21,Hornbeck Offshore Services (Withdrawn),Other,100.0000
23,Coolbit Technologies Ltd. (Withdrawn),Technologies,22.5000
32,Timber Road Acquisition Corp. (Withdrawn),Acquisition Corp,200.0000
45,pan-Africa Corp. (Withdrawn),Acquisition Corp,125.0000
61,Living Homeopathy (Withdrawn),Other,18.7500
105,Nuclea Energy (Withdrawn),Other,50.4000
136,APEX Global Solutions Ltd. (WITHDRAWN),Limited,15.0000
141,Cloud Data Holdings (Withdrawn),Holdings,15.9375


### Aggregation & Answer

In [7]:
company_type_totals = (
    withdrawn_df.groupby("Company Type")["Shares_offered_value"]
    .sum()
    .sort_values(ascending=False)
)
company_type_totals

Company Type
Acquisition Corp    499.9850
Inc.                351.0000
Holdings            311.6575
Other               290.4450
Limited             203.8500
Technologies        184.9000
Group                32.5000
Name: Shares_offered_value, dtype: float64

In [8]:
top_class = company_type_totals.idxmax()
top_value = company_type_totals.max()

print(f"Highest total withdrawn value: '{top_class}' class with ${top_value:,.1f} million")

Highest total withdrawn value: 'Acquisition Corp' class with $500.0 million


---

## Question 2: [IPO] Median Sharpe Ratio for 2025 IPOs (First 8 Months)

**What is the median Sharpe ratio (as of 11 September 2026) for companies that went public before 1 September 2025?**

### Steps
1. Download the list of 2025 IPOs from [iposcoop.com/2025-pricings](https://www.iposcoop.com/2025-pricings/).
2. Filter to IPOs with `Offer Date` before 2025-09-01, excluding 0% return rows.
3. Download daily OHLCV data for the filtered tickers via `yfinance`, skipping tickers with no data (likely delisted).
4. Compute `growth_252d`, 30-day annualized `volatility`, and `Sharpe` ratio (risk-free rate 5.0%).
5. Snapshot the data on `2026-09-11` and compute descriptive statistics.

### Data Loading

In [9]:
ipo_2025_url = "https://www.iposcoop.com/2025-pricings/"

response_2025 = requests.get(ipo_2025_url, headers=HEADERS, timeout=15)
response_2025.raise_for_status()

ipo_2025_df = pd.read_html(StringIO(response_2025.text))[0].copy()
print(ipo_2025_df.shape)

(231, 10)


### Filtering

In [10]:
ipo_2025_df["Offer Date"] = pd.to_datetime(ipo_2025_df["Offer Date"])

filtered_ipos_df = ipo_2025_df[
    (ipo_2025_df["Offer Date"] < "2025-09-01") & (ipo_2025_df["Return"] != "0.00%")
].copy()

print("Filtered IPOs:", len(filtered_ipos_df))
tickers = filtered_ipos_df["Symbol"].tolist()

Filtered IPOs: 146


### Data Download

In [11]:
import yfinance as yf

raw_prices = yf.download(
    tickers, start="2024-06-01", group_by="ticker", threads=True, progress=False, auto_adjust=False
)

price_frames = []
missing_tickers = []
for ticker in tickers:
    try:
        ticker_prices = raw_prices[ticker].dropna(how="all").copy()
    except KeyError:
        ticker_prices = pd.DataFrame()

    if ticker_prices.empty:
        missing_tickers.append(ticker)
        continue

    ticker_prices = ticker_prices.reset_index()
    ticker_prices["Ticker"] = ticker
    price_frames.append(ticker_prices[["Date", "Ticker", "Close"]])

stocks_df = pd.concat(price_frames, ignore_index=True).sort_values(["Ticker", "Date"])

print("Tickers with data:", stocks_df["Ticker"].nunique())
print("Tickers missing/delisted:", len(missing_tickers))

HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: MJID"}}}


$TBH: possibly delisted; no timezone found


$MCTR: possibly delisted; no timezone found


HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: SDM"}}}


$MJID: possibly delisted; no timezone found


...

$EPWK: possibly delisted; no timezone found


$EMPG: possibly delisted; no timezone found



13 Failed downloads:


['TBH', 'MCTR', 'MJID', 'SDM', 'PTNM', 'AGH', 'CAEP', 'CEPT', 'MTSR', 'AHL', 'SKBL', 'EPWK', 'EMPG']: possibly delisted; no timezone found


Tickers with data: 133
Tickers missing/delisted: 13


### Feature Engineering

In [12]:
stocks_df["growth_252d"] = stocks_df.groupby("Ticker")["Close"].transform(
    lambda close: close / close.shift(252)
)
stocks_df["volatility"] = stocks_df.groupby("Ticker")["Close"].transform(
    lambda close: close.rolling(30).std() * np.sqrt(252)
)

### Sharpe Ratio Calculation

In [13]:
RISK_FREE_RATE = 0.05

stocks_df["Sharpe"] = (stocks_df["growth_252d"] - RISK_FREE_RATE) / stocks_df["volatility"]

A few stocks have (near-)zero 30-day rolling volatility — prices frozen flat, likely halted/stale quotes — which makes `Sharpe = (growth_252d - 0.05) / volatility` blow up. Floating-point noise means this doesn't always land on exactly `0` (e.g. `1e-6` instead), so a small tolerance is used rather than an exact `== 0` check. This corrupts the mean/std in `describe()` even though the median is barely affected, so these are treated as undefined (`NaN`) before the final analysis rather than as real risk-adjusted returns.

In [14]:
VOLATILITY_EPS = 1e-4  # treat near-zero rolling std (flat/stale prices) as zero
negligible_volatility = stocks_df["volatility"].abs() < VOLATILITY_EPS
print("Rows with negligible volatility (Sharpe -> inf/extreme):", negligible_volatility.sum())

stocks_df.loc[negligible_volatility, "Sharpe"] = np.nan
stocks_df["Sharpe"] = stocks_df["Sharpe"].replace([np.inf, -np.inf], np.nan)

Rows with negligible volatility (Sharpe -> inf/extreme): 752


### Final Analysis

In [15]:
snapshot_df = stocks_df[stocks_df["Date"] == "2026-09-11"].copy()

print("Stocks with data on 2026-09-11:", len(snapshot_df))
print("Stocks that reached the 252-day milestone:", snapshot_df["growth_252d"].notna().sum())

snapshot_df[["growth_252d", "Sharpe"]].describe()

Stocks with data on 2026-09-11: 132
Stocks that reached the 252-day milestone: 130


Price,growth_252d,Sharpe
count,130.000000,127.000000
mean,1.052117,0.184883
std,3.049217,0.578563
min,0.001005,-0.040147
25%,0.141035,0.010665
50%,0.594534,0.046432
75%,1.038821,0.119233
max,33.638270,5.385326


In [16]:
median_sharpe = snapshot_df["Sharpe"].median()
median_growth = snapshot_df["growth_252d"].median()
mean_growth = snapshot_df["growth_252d"].mean()

print(f"Median growth_252d: {median_growth:.3f} | Mean growth_252d: {mean_growth:.3f}")
print(f"Median Sharpe ratio: {median_sharpe:.4f}")

Median growth_252d: 0.595 | Mean growth_252d: 1.052
Median Sharpe ratio: 0.0464


---

## Question 3: [IPO] 'Fixed Months Holding Strategy'

**What is the optimal number of months (1 to 12) to hold a newly IPO'd stock in order to maximize the median growth value?**

Goal: investigate the performance of the 2025 IPO stocks (from Question 2) over fixed holding periods
of 1-12 months, to find which horizon yields the highest **median** return relative to the first
day's closing price.

### Steps
1. Reuse `stocks_df` (daily OHLCV + features for the Question 2 IPO tickers).
2. Add 12 future growth columns `future_growth_1m` ... `future_growth_12m` (1 month = 21 trading days).
3. For each ticker, find its first trading day (`min_date`) and closing price.
4. Inner-join the `min_date` rows with the future-growth columns to isolate each stock's return path starting from its IPO date.
5. Compute descriptive statistics across the 12 columns and find the month with the highest median (50th percentile).

### Feature Engineering: Future Growth Columns

In [17]:
future_growth_days = [months * 21 for months in range(1, 13)]

for months, days in zip(range(1, 13), future_growth_days):
    col_name = f"future_growth_{months}m"
    stocks_df[col_name] = stocks_df.groupby("Ticker")["Close"].transform(
        lambda close, d=days: close.shift(-d) / close
    )

future_growth_cols = [f"future_growth_{months}m" for months in range(1, 13)]
stocks_df[["Ticker", "Date", "Close"] + future_growth_cols].head()

Price,Ticker,Date,Close,future_growth_1m,future_growth_2m,future_growth_3m,future_growth_4m,future_growth_5m,future_growth_6m,future_growth_7m,future_growth_8m,future_growth_9m,future_growth_10m,future_growth_11m,future_growth_12m
40382,AAPG,2025-01-24,17.379999,1.159954,1.056962,1.512946,1.485328,2.348101,2.259494,2.459436,2.176640,2.040276,1.842923,1.604143,1.472382
40383,AAPG,2025-01-27,17.250000,1.198261,1.171015,1.466087,1.416000,2.295652,2.323478,2.400000,2.164058,2.003478,1.876522,1.604638,1.466667
40384,AAPG,2025-01-28,17.200001,1.151163,1.258721,1.406977,1.473837,2.289244,2.180232,2.391453,2.241279,1.947674,1.867442,1.506977,1.453488
40385,AAPG,2025-01-29,17.330000,1.098096,1.266590,1.409694,1.407963,2.249856,2.135026,2.450664,2.252741,1.927294,1.875361,1.504328,1.395268
40386,AAPG,2025-01-30,17.600000,1.060795,1.309091,1.428977,1.432954,2.264204,2.113636,2.376136,2.248864,1.859091,1.846591,1.490909,1.316477


### Identify Entry Points & Data Alignment

In [18]:
min_date_df = (
    stocks_df.groupby("Ticker", as_index=False)["Date"]
    .min()
    .rename(columns={"Date": "min_date"})
)

entry_df = stocks_df.merge(
    min_date_df, left_on=["Ticker", "Date"], right_on=["Ticker", "min_date"], how="inner"
)

print("Tickers:", stocks_df["Ticker"].nunique())
print("Entry (IPO day) records:", len(entry_df))
entry_df[["Ticker", "Date", "Close"] + future_growth_cols].head()

Tickers: 133
Entry (IPO day) records: 133


,Ticker,Date,Close,future_growth_1m,future_growth_2m,future_growth_3m,future_growth_4m,future_growth_5m,future_growth_6m,future_growth_7m,future_growth_8m,future_growth_9m,future_growth_10m,future_growth_11m,future_growth_12m
0,AAPG,2025-01-24,17.379999,1.159954,1.056962,1.512946,1.485328,2.348101,2.259494,2.459436,2.176640,2.040276,1.842923,1.604143,1.472382
1,AARD,2025-02-13,14.310000,0.647100,0.555556,0.635220,0.758211,0.835779,0.753319,0.716981,0.973445,0.758910,1.011181,1.072676,0.870021
2,ADVB,2025-03-06,73.000000,0.920548,0.523288,0.255069,0.181370,0.099178,0.114521,0.132329,0.124932,0.104110,0.095890,0.065479,0.067808
3,AII,2025-05-08,16.900000,1.005917,1.020118,1.039645,1.156213,1.414793,1.469231,1.156213,1.196450,1.049112,1.124852,1.079290,1.171006
4,AIRO,2025-06-13,24.000000,1.085417,0.955833,0.841667,0.836667,0.557083,0.354167,0.551250,0.383333,0.442500,0.360417,0.265417,0.329167


### Median Analysis

In [19]:
growth_stats = entry_df[future_growth_cols].describe()
growth_stats.loc[["count", "mean", "50%", "max"]]

,future_growth_1m,future_growth_2m,future_growth_3m,future_growth_4m,future_growth_5m,future_growth_6m,future_growth_7m,future_growth_8m,future_growth_9m,future_growth_10m,future_growth_11m,future_growth_12m
count,132.000000,131.000000,131.000000,131.000000,131.000000,131.000000,131.000000,131.000000,131.000000,131.000000,131.000000,130.000000
mean,1.061827,1.206651,1.174116,1.052652,1.008658,1.048758,1.079424,22.170111,15.974968,12.291991,5.848349,12.872358
50%,0.935351,0.890411,0.827160,0.741678,0.698750,0.726000,0.662000,0.609756,0.590216,0.533640,0.492453,0.529827
max,8.787346,16.066783,16.344464,11.386643,8.182777,9.894552,20.741478,2777.777667,1944.444367,1458.333275,630.555530,1549.999938


In [20]:
median_by_month = growth_stats.loc["50%"]
mean_by_month = growth_stats.loc["mean"]

best_col = median_by_month.idxmax()
best_month = int(best_col.replace("future_growth_", "").replace("m", ""))
best_median = median_by_month.max()

print(f"Optimal holding period: {best_month} month(s), median growth = {best_median:.3f}")
print()
print("Median growth by holding period (months 1-12):")
print(median_by_month.values.round(3))
print()
print("Mean growth by holding period (months 1-12), for comparison:")
print(mean_by_month.values.round(3))

Optimal holding period: 1 month(s), median growth = 0.935

Median growth by holding period (months 1-12):
[0.935 0.89  0.827 0.742 0.699 0.726 0.662 0.61  0.59  0.534 0.492 0.53 ]

Mean growth by holding period (months 1-12), for comparison:
[ 1.062  1.207  1.174  1.053  1.009  1.049  1.079 22.17  15.975 12.292
  5.848 12.872]


### Observations

- The **median** growth trends *downward* as the holding period lengthens — the best median outcome is
  at the shortest horizon, and it degrades the longer you hold. For a typical (median) 2025 IPO stock,
  waiting longer has not paid off; the data does not support a "buy and hold newly IPO'd stocks for a
  year" strategy.
- The **mean** tells a very different story at longer horizons (e.g. months 8-12), where it spikes far
  above the median and even above 1.0 by large multiples. This is driven by a handful of extreme
  outliers (occasional penny-stock-style spikes of hundreds or thousands of percent), not by the typical
  stock performing well. This is the same mean-vs-median distortion seen in Question 2: a few big winners
  make the average look attractive while most stocks are flat-to-down.

---

## Question 4: [Strategy] Simple RSI-Based Trading Strategy

**What is the total profit (in $ thousands) you would have earned by investing $1000 every time a stock was oversold (RSI < 30)?**

Goal: apply a simple rule-based trading strategy using the RSI technical indicator to identify
oversold signals, and calculate cumulative profit over a 25-year period (2000-01-01 to 2025-06-01).

### Steps
1. Data Acquisition: download the precomputed technical/macro indicators dataset from Google Drive via `gdown`.
2. Strategy Setup: an "oversold" signal is `rsi < 30`.
3. Filtering: keep only rows between 2000-01-01 and 2025-06-01 where the signal triggers.
4. Profit Calculation: invest $1,000 per signal, using `growth_future_30d` as the 30-day outcome.
   `net_income = 1000 * (selected_df['growth_future_30d'] - 1).sum()`

### Data Acquisition

In [21]:
import gdown

file_id = "1grCTCzMZKY5sJRtdbLVCXg8JXA8VPyg-"
gdown.download(f"https://drive.google.com/uc?id={file_id}", "data.parquet", quiet=False)

indicators_df = pd.read_parquet(
    "data.parquet", engine="pyarrow", columns=["Date", "Ticker", "rsi", "growth_future_30d"]
)

print(indicators_df.shape)
indicators_df.head()

Downloading...
From (original): https://drive.google.com/uc?id=1grCTCzMZKY5sJRtdbLVCXg8JXA8VPyg-
From (redirected): https://drive.google.com/uc?id=1grCTCzMZKY5sJRtdbLVCXg8JXA8VPyg-&confirm=t&uuid=ce65f296-e8c4-4b7d-bd33-f0f0fec5bb27
To: C:\Users\Siddharth\Downloads\Stock-Markets-Analytics-Zoomcamp-2026\Week_2\data.parquet



  0%|          | 0.00/130M [00:00<?, ?B/s]


  0%|          | 524k/130M [00:00<00:36, 3.59MB/s]


  2%|▏         | 3.15M/130M [00:00<00:08, 14.7MB/s]


  5%|▌         | 6.82M/130M [00:00<00:09, 13.3MB/s]

...


 94%|█████████▍| 123M/130M [00:05<00:00, 30.5MB/s]


 97%|█████████▋| 127M/130M [00:05<00:00, 32.6MB/s]


100%|██████████| 130M/130M [00:05<00:00, 22.4MB/s]

(229932, 4)


,Date,Ticker,rsi,growth_future_30d
0,1986-03-13,MSFT,NaN,0.982144
1,1986-03-14,MSFT,NaN,0.922418
2,1986-03-17,MSFT,NaN,0.881354
3,1986-03-18,MSFT,NaN,0.921744
4,1986-03-19,MSFT,NaN,0.964604


### Strategy Setup & Filtering

In [22]:
RSI_THRESHOLD = 30

selected_df = indicators_df[
    (indicators_df["rsi"] < RSI_THRESHOLD)
    & (indicators_df["Date"] >= "2000-01-01")
    & (indicators_df["Date"] <= "2025-06-01")
].dropna(subset=["growth_future_30d"])

print("Number of trades:", len(selected_df))
selected_df.head()

Number of trades: 5206


,Date,Ticker,rsi,growth_future_30d
3562,2000-04-14,MSFT,28.599774,0.898820
3567,2000-04-24,MSFT,28.183709,1.102251
3667,2000-09-14,MSFT,28.744488,0.975309
3668,2000-09-15,MSFT,24.777933,0.985394
3669,2000-09-18,MSFT,22.350677,0.972222


### Profit Calculation & Answer

In [23]:
net_income = 1000 * (selected_df["growth_future_30d"] - 1).sum()

print(f"Number of trades: {len(selected_df):,}")
print(f"Net income: ${net_income:,.0f}")
print(f"Net income (in $ thousands): ${net_income / 1000:,.1f}K")

Number of trades: 5,206
Net income: $65,806
Net income (in $ thousands): $65.8K


---

## Q5. [Exploratory, Optional] Predicting a Positive-Return IPO

**How would you change the strategy to increase profitability?**

### Momentum-Confirmation Filter
Don't buy at the IPO. Instead:
1. **Screen**: wait 3 months after listing. Only keep IPOs whose price is *above* their IPO-day close (positive 3-month return) — this discards the early losers seen in Q3/Q4.
2. **Enter**: invest $1,000 at the 3-month price for the surviving "eligible" IPOs only.
3. **Track**: measure return/value at +6m, +9m, +12m from that entry point (using latest available price if the checkpoint is still in the future).

**Why this should help**: early positive momentum acts as a proxy for genuine demand/fundamentals, filtering out the flops that drag down the median in Q3. It trades "buy every IPO" for "buy only the IPOs the market has already validated," at the cost of missing the very first 3 months of any winner's move.